<a href="https://colab.research.google.com/github/warrensuca/chinese-recipe-api/blob/main/warrensuca/chinese-recipe-api/cleaning/recipe_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Combine, Clean, and Normalize the Recipe Data
The point of the scraping is to have recipe data useful for k-means clustering and a kNN model. Therefore, the data should be combined, cleaned, and normalized.

Clean by removing duplicates and expanding individual nutrition facts into their own columns

Normalize by looking at the distributions.

In [120]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
print("setup complete")

setup complete


#Data Cleaning

##Woks of Life cleaning
We are looking to
- remove duplicates
- parse and expand nutritional info into individual columns

In [121]:
df_wol = pd.read_csv("woks_of_life_recipes.csv")
print("original shape:", df_wol.shape)


df_wol = df_wol[~(df_wol == "[]").any(axis=1)]

df_wol = df_wol[df_wol['Nutrition_Facts'].str.len() != 5] # debug trial and error: print(x, len(x)) -> ['4'] 5
print("new shape:", df_wol.shape)

def format_serving(stringified_list):

    nutrition_list = ast.literal_eval(stringified_list)

    end = "serving" if nutrition_list[0] == 1 else "servings"

    nutrition_list[0] = f"Serving: {nutrition_list[0]}{end}"


    return str(nutrition_list)


df_wol['Nutrition_Facts'] = df_wol['Nutrition_Facts'].map(format_serving)

nutrition_columns = ['Calories', 'Carbohydrates', 'Protein', 'Fat', 'Saturated Fat', 'Sodium', 'Sugar']
valid_nutrients = set(nutrition_columns)

suffix_to_value = {
    'kcal': 1,
    'mg' : 1/1000,
    'g' : 1,
    'kg' : 1000
}

all_nutrition_data = []
for nutrition_string in df_wol['Nutrition_Facts']:

  nutrition_list = ast.literal_eval(nutrition_string)

  temp_dict = {col: 0 for col in nutrition_columns}


  for nutrient_profile in nutrition_list:


      if ':' not in nutrient_profile:
          continue

      name, value_str = nutrient_profile.split(':', 1)
      nutrient_name = name.strip()

      if nutrient_name in valid_nutrients:
          num = ""
          suffix = ""

          for char in value_str:
              if char.isdigit() or char == '.':
                  num += char
              elif char == '(':
                  break
              else:
                  suffix += char

          if num: # ensure we actually found a number
              temp_dict[nutrient_name] += float(num) * suffix_to_value[suffix.strip()]

  all_nutrition_data.append(temp_dict)
df_wol['Nutrition_Facts'] = df_wol['Nutrition_Facts']
nutrition_df = pd.DataFrame(all_nutrition_data)


df_wol = df_wol.reset_index(drop=True)

# glue the new columns onto the right side of original DataFrame
df_wol = pd.concat([df_wol, nutrition_df], axis=1)

print("Final shape:", df_wol.shape)
display(df_wol.head())


original shape: (1414, 5)
new shape: (1266, 5)
Final shape: (1266, 12)


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts,Calories,Carbohydrates,Protein,Fat,Saturated Fat,Sodium,Sugar
0,shallot-vinaigrette,"['1 large shallot (minced)', '2 tablespoons h...","['large shallot', 'hot water', 'white wine vin...","['In a measuring cup, add the shallots and hot...","['Serving: 6servings', 'Calories: 173kcal (9%)...",173.0,3.0,0.3,18.0,2.0,0.408,2.0
1,sesame-tofu,['2 blocks firm or extra firm tofu (anywhere f...,"['firm or extra firm tofu', 'cornstarch', 'un-...","['Drain each block of tofu. Using your hands, ...","['Serving: 6servings', 'Calories: 358kcal (18%...",358.0,29.0,15.0,20.0,2.0,0.592,9.0
2,pork-belly-char-siu,['2 pounds boneless skinless pork belly (if tr...,"['boneless skinless pork belly', 'granulated s...","['If you have a larger piece of pork belly, cu...","['Serving: 6servings', 'Calories: 682kcal (34%...",682.0,11.0,15.0,81.0,29.0,0.785,10.0
3,red-bean-zongzi,['2 pounds sweet rice (also known as sticky ri...,"['sweet rice', 'lye water', 'neutral oil', 'dr...","['In a large, fine mesh strainer, rinse the sw...","['Serving: 12servings', 'Calories: 344kcal (17...",344.0,72.0,8.0,2.0,0.2,0.006,0.0
4,stir-fry-spinach,['1 pound spinach (preferably Taiwanese spinac...,"['spinach', 'neutral oil', 'garlic', 'Salt to ...",['Wash spinach clean by soaking a few times an...,"['Serving: 4servings', 'Calories: 130kcal (7%)...",130.0,5.0,3.0,12.0,1.0,0.235,0.5


##Omnivore's Cookbook cleaning
We are looking to
- remove duplicates
- parse and expand nutritional info into individual columns

In [122]:
df_oc = pd.read_csv('omnivores_cookbook_recipes.csv')

print("original shape:", df_oc.shape)


df_oc = df_oc[~(df_oc == "[]").any(axis=1)]

print("new shape:", df_oc.shape)
display(df_oc)

all_nutrition_data = []
for nutrition_string in df_oc['Nutrition_Facts']:

  nutrition_list = ast.literal_eval(nutrition_string)[1:]

  #print(nutrition_list)

  temp_dict = {col: 0 for col in nutrition_columns}
  for nutrient_profile in nutrition_list:


      if ':' not in nutrient_profile:
          continue

      name, value_str = nutrient_profile.split(':', 1)
      nutrient_name = name.strip()

      if nutrient_name in valid_nutrients:
          num = ""
          suffix = ""

          for char in value_str:
              if char.isdigit() or char == '.':
                  num += char
              elif char == '(' or char == '-':
                  break
              else:
                  suffix += char

          if num: # ensure we actually found a number
              temp_dict[nutrient_name] += float(num) * suffix_to_value[suffix.strip()]

  all_nutrition_data.append(temp_dict)

nutrition_df = pd.DataFrame(all_nutrition_data)


df_oc = df_oc.reset_index(drop=True)

# glue the new columns onto the right side of original DataFrame
df_oc = pd.concat([df_oc, nutrition_df], axis=1)

print("Final shape:", df_oc.shape)

display(df_oc.head())

original shape: (684, 5)
new shape: (637, 5)


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts
0,thai-fish-cakes,"['1 pound white fish fillet , cut into 1-inch ...","['white fish fillet', 'red curry paste', 'larg...","['Fish cake:', 'Combine the fish, curry paste,...","['Serving: 1of the 6 servings', 'Calories: 308..."
1,budae-jjigae,"['1 tablespoon peanut oil (or vegetable oil)',...","['peanut oil', 'ground beef', 'green onions', ...",['Heat oil in a 4-quart dutch oven (or heavy p...,"['Serving: 4servings', 'Calories: 522kcal', 'C..."
2,drunken-chicken,['2 bone-in skin-on chicken leg quarters (incl...,"['bone-in skin-on chicken leg quarters', 'salt...",['Combine the salt and Sichuan peppercorns in ...,"['Serving: 1serving', 'Calories: 99kcal', 'Car..."
3,matcha-tiramisu,['2 tablespoons ceremonial grade matcha and ex...,"['ceremonial grade matcha', 'very hot water', ...",['To make the matcha soak: Sift matcha into a ...,"['Serving: 1serving', 'Calories: 224kcal', 'Ca..."
4,clams-in-black-bean-sauce,"['2 lbs manila clams', 'Salt , for soaking the...","['manila clams', 'Salt', 'vegetable oil', 'Sha...",['Place the clams in a large bowl. Add 8 cups ...,"['Serving: 1serving', 'Calories: 318kcal', 'Ca..."
...,...,...,...,...,...
674,how-to-make-red-bean-paste,"['300 grams (10 ounces) azuki beans', '200 to ...","['azuki beans', 'rock sugar', 'salt']",['Combine red beans and 3 cups of water in pre...,"['Serving: 42g', 'Calories: 147kcal', 'Carbohy..."
675,baked-seafood-pasta,"['2 teaspoons white rum', '200 grams (7 ounces...","['white rum', 'shrimp', 'squid', 'sea salt', '...","['In a small bowl, add the shrimp, 1 teaspoon ...","['Serving: 423g', 'Calories: 790kcal', 'Carboh..."
677,beef-burger-and-kimchi-pizza,['1 12 ” (30 cm) thin pizza crusts (or pizza d...,"['thin pizza crusts', 'sesame oil', 'minced ki...",['Heat the sesame oil in a small saucepan over...,"['Serving: 1serving', 'Calories: 350kcal', 'Ca..."
678,moms-best-beef-stew-with-tendon,"['2.4 kilograms (5 pounds) beef plate , cut in...","['beef plate', 'beef tendon', 'Shaoxing wine',...",['Place beef tendon in a pressure cooker and a...,"['Serving: 169g', 'Calories: 299kcal', 'Carboh..."


Final shape: (637, 12)


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts,Calories,Carbohydrates,Protein,Fat,Saturated Fat,Sodium,Sugar
0,thai-fish-cakes,"['1 pound white fish fillet , cut into 1-inch ...","['white fish fillet', 'red curry paste', 'larg...","['Fish cake:', 'Combine the fish, curry paste,...","['Serving: 1of the 6 servings', 'Calories: 308...",308.0,21.8,16.3,16.5,3.9,1.053,10.9
1,budae-jjigae,"['1 tablespoon peanut oil (or vegetable oil)',...","['peanut oil', 'ground beef', 'green onions', ...",['Heat oil in a 4-quart dutch oven (or heavy p...,"['Serving: 4servings', 'Calories: 522kcal', 'C...",522.0,26.6,39.1,29.1,8.3,1.353,10.1
2,drunken-chicken,['2 bone-in skin-on chicken leg quarters (incl...,"['bone-in skin-on chicken leg quarters', 'salt...",['Combine the salt and Sichuan peppercorns in ...,"['Serving: 1serving', 'Calories: 99kcal', 'Car...",99.0,3.0,9.9,5.1,1.4,0.270,2.0
3,matcha-tiramisu,['2 tablespoons ceremonial grade matcha and ex...,"['ceremonial grade matcha', 'very hot water', ...",['To make the matcha soak: Sift matcha into a ...,"['Serving: 1serving', 'Calories: 224kcal', 'Ca...",224.0,25.5,7.0,10.8,5.8,0.135,12.6
4,clams-in-black-bean-sauce,"['2 lbs manila clams', 'Salt , for soaking the...","['manila clams', 'Salt', 'vegetable oil', 'Sha...",['Place the clams in a large bowl. Add 8 cups ...,"['Serving: 1serving', 'Calories: 318kcal', 'Ca...",318.0,43.1,3.9,15.0,2.9,2.364,17.2


##Combine the two partially cleaned datasets
Then use .describe() to use identify extreme outliers among other things

In [123]:
df_final = pd.concat([df_oc, df_wol], axis=0, ignore_index=True)
display(df_final.sample(3))
df_final.describe()

,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts,Calories,Carbohydrates,Protein,Fat,Saturated Fat,Sodium,Sugar
301,mango-sticky-rice,"['1 cup uncooked sticky rice', '1 can full-fa...","['uncooked sticky rice', 'full-fat coconut mil...",['Rinse the sticky rice with tap water and use...,"['Serving: 1serving', 'Calories: 288kcal', 'Ca...",288.0,42.4,3.1,12.9,11.1,0.276,24.9
1801,roast-chicken-wild-mushroom-sticky-rice-risotto,"['4 chicken leg quarters (or 8 chicken thighs,...","['chicken leg quarters', 'salt and pepper', 'l...",['Take your chicken leg quarters and pat them ...,"['Serving: 8servings', 'Calories: 387kcal (19%...",387.0,37.0,21.0,17.0,5.0,0.497,3.0
169,sichuan-boiled-beef,"['1 lb beef flank steak', '1 tablespoon Shaoxi...","['beef flank steak', 'Shaoxing wine', 'soy sau...",['Cut the beef flank steak along the grain int...,"['Serving: 1serving', 'Calories: 357kcal', 'Ca...",357.0,13.0,27.4,21.5,3.1,1.047,7.6


,Calories,Carbohydrates,Protein,Fat,Saturated Fat,Sodium,Sugar
count,1903.000000,1903.000000,1903.000000,1903.000000,1903.000000,1903.000000,1903.000000
mean,307.583085,25.589080,16.340026,15.863537,5.050382,0.582260,5.955302
std,170.962878,21.771941,14.740488,12.217959,5.694472,0.434239,7.036031
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,186.000000,9.000000,6.000000,8.000000,1.200000,0.298000,2.000000
50%,288.000000,19.000000,14.000000,13.600000,3.000000,0.540000,3.800000
75%,408.000000,39.000000,23.950000,21.000000,7.000000,0.818000,7.500000
max,1338.000000,204.000000,315.000000,103.000000,82.000000,8.110000,75.000000


##Spot extreme outliers and adjust them

In [126]:
display(df_final[df_final['Protein'] == 315]) #zucchini stir fry is more like 3.15 instead of 315

df_final.loc[285, 'Protein'] = 3.15
df_final.loc[285, 'Nutrition_Facts'] = df_final.loc[285, 'Nutrition_Facts'].replace('315', '3.15')

display(df_final.loc[285])

display(df_final[df_final['Carbohydrates'] == 204])  #recipe didn't scale to 1 serving, #153

df_final.loc[153, 'Carbohydrates'] = 51
df_final.loc[153, 'Nutrition_Facts'] = df_final.loc[153, 'Nutrition_Facts'].replace('204', '51')

display(df_final.loc[153])


display(df_final[df_final['Sodium'] == 8.11]) #recipe didn't scale to 1 serving, #953
df_final.loc[953, 'Sodium'] = 2.03
df_final.loc[953, 'Nutrition_Facts'] = df_final.loc[953, 'Nutrition_Facts'].replace('8.11', '2.03')
display(df_final.loc[953])

display(df_final[df_final['Sugar'] == 75]) #recipe didn't scale to 1 serving #1530
df_final.loc[1530, 'Carbohydrates'] = 3.75
df_final.loc[1530, 'Nutrition_Facts'] = df_final.loc[1530, 'Nutrition_Facts'].replace('75', '3.75')
display(df_final.loc[1530])

,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts,Calories,Carbohydrates,Protein,Fat,Saturated Fat,Sodium,Sugar


,285
Name,zucchini-stir-fry
Ingredients_List,"['1 lb zucchini', '1 teaspoon salt (Optional)'..."
Ingredients_Names,"['zucchini', 'salt', 'peanut oil', 'dried chil..."
Procedure,['Remove the tough ends from the zucchini. Hal...
Nutrition_Facts,"['Serving: 1serving', 'Calories: 53kcal', 'Car..."
Calories,53.0
Carbohydrates,4.9
Protein,3.15
Fat,3.6
Saturated Fat,0.6


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts,Calories,Carbohydrates,Protein,Fat,Saturated Fat,Sodium,Sugar


,153
Name,spam-fried-rice
Ingredients_List,['1 tablespoon Shaoxing wine (or chicken broth...
Ingredients_Names,"['Shaoxing wine', 'soy sauce', 'oyster sauce',..."
Procedure,['Mix the sauce ingredients together in a smal...
Nutrition_Facts,"['Serving: 1serving', 'Calories: 526kcal', 'Ca..."
Calories,526.0
Carbohydrates,51.0
Protein,15.9
Fat,22.2
Saturated Fat,6.0


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts,Calories,Carbohydrates,Protein,Fat,Saturated Fat,Sodium,Sugar


,953
Name,spicy-stir-fried-thai-basil-clams
Ingredients_List,['3 pounds littleneck or cherrystone clams (ab...
Ingredients_Names,"['littleneck or cherrystone clams', 'cold wate..."
Procedure,"['To purge the clams of sand and grit, fill a ..."
Nutrition_Facts,"['Serving: 4servings', 'Calories: 132kcal (7%)..."
Calories,132.0
Carbohydrates,6.0
Protein,8.0
Fat,8.0
Saturated Fat,1.0


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts,Calories,Carbohydrates,Protein,Fat,Saturated Fat,Sodium,Sugar
1530,pumpkin-cupcakes,"['2 cups all-purpose flour', '1 teaspoon bakin...","['all-purpose flour', 'baking soda', 'baking p...","['In a medium bowl, combine the flour, baking ...","['Serving: 20servings', 'Calories: 567kcal (28...",567.0,3.75,3.0,24.0,14.0,0.196,75.0


,1530
Name,pumpkin-cupcakes
Ingredients_List,"['2 cups all-purpose flour', '1 teaspoon bakin..."
Ingredients_Names,"['all-purpose flour', 'baking soda', 'baking p..."
Procedure,"['In a medium bowl, combine the flour, baking ..."
Nutrition_Facts,"['Serving: 20servings', 'Calories: 567kcal (28..."
Calories,567.0
Carbohydrates,3.75
Protein,3.0
Fat,24.0
Saturated Fat,14.0
